## Использование линейной регрессии для построения причинно-следственных выводов

**Материал подготовлен для курса [Объяснимое машинное обучение](https://ai-interpretability.school).**

Стараясь оценить связь признака с целевой переменной, исследовательницы часто прибегают к анализу коэффициента корреляции. Кроме того, коэффициент корреляции — это первое, с чем знакомят на курсах Data science, так что знание о нем есть даже у начинающих. Однако:

> *Correlation is not causation.*

или «корреляция не равна причинно-следственной связи». Но как быть, если причинно-следственную связь оценить все же хочется?

В модуле курса «Линейные интерпретируемые модели» коэффициенты регрессии служат для анализа важности признаков. Здесь мы посмотрим, когда коэффициент регрессии можно читать как причинный эффект, а когда нельзя, и чем его дополняют: контролем по наблюдаемым переменным, DML и инструментальной переменной.

Чтобы материал был понятен каждому, начнем с самого начала.

In [ ]:
#Импорт библиотек
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_predict

Оценим, как готовность получить высшее образование связана с оценкой по математике.

**Математика**

Пусть:\
$T$ — признак, влияние которого мы хотим оценить: готовность получить высшее образование ($T=1$, если студентка готова его получать, $T=0$, если нет)\
$Y_0$ — оценка студентки, если она не готова получать высшее образование\
$Y_1$ — оценка студентки, если она готова получать высшее образование\
$Y$ — наблюдаемая оценка: $Y = T\,Y_1 + (1-T)\,Y_0$, то есть $Y=Y_1$ при $T=1$ и $Y=Y_0$ при $T=0$\
$TE = Y_1 - Y_0$ — эффект воздействия: насколько меняется оценка при переходе признака с 0 на 1

In [ ]:
#В идеальном мире наш набор данных выглядел бы так (данные придуманы для примера)
ate_att_data = pd.DataFrame(dict(
      i= [1,2,3,4],
      Y0=[500,600,800,700],
      Y1=[450,600,600,750],
      T= [0,0,1,1],
      Y= [500,600,600,750],
      TE=[-50,0,-200,50],
))
ate_att_data

In [ ]:
#Но в реальном он выглядит так: у каждой студентки известен только один из двух исходов
pd.DataFrame(dict(
    i= [1,2,3,4],
    Y0=[500,600,np.nan,np.nan],
    Y1=[np.nan,np.nan,600,750],
    T= [0,0,1,1],
    Y= [500,600,600,750],
    TE=[np.nan,np.nan,np.nan,np.nan],
))

Студенток у нас много. Для удобства введем еще пару вещей:

- $Y_{i0}$ — потенциальная оценка $i$-й студентки при неготовности получать высшее образование
- $Y_{i1}$ — потенциальная оценка $i$-й студентки при готовности получать высшее образование

- Тогда влияние признака «готовность получить высшее образование» можно определить как: $Y_{i1} - Y_{i0}$


- $ATE$, *average treatment effect*, — средний эффект наличия признака
- $ATT$, *average treatment effect on treated*, — средний эффект наличия признака для наблюдений, у которых он есть

Заметим, что:\
$ATE = E[Y_1 - Y_0]$ — матожидание от разности целевого значения с наличием и отсутствием признака\
$ATT = E[Y_1 - Y_0|T=1]$ — матожидание от разности целевого значения с наличием и отсутствием признака при условии, что признак равен 1.

По идеальному набору данных мы смогли бы рассчитать все по определению. По реальному — нет. И так как мы решили работать с реальностью, рассмотрим:

$$E[Y|T=1] - E[Y|T=0] = E[Y_1|T=1] - E[Y_0|T=0] = E[Y_1|T=1] - E[Y_0|T=1] + E[Y_0|T=1] - E[Y_0|T=0]=E[Y_1-Y_0|T=1] + (E[Y_0|T=1] - E[Y_0|T=0])$$


Получили: $E[Y_1-Y_0|T=1] + (E[Y_0|T=1] - E[Y_0|T=0])$

Первое слагаемое $E[Y_1-Y_0|T=1]$ — это ATT, средний эффект воздействия на воздействованных: то, что мы и хотим измерить. Второе $(E[Y_0|T=1] - E[Y_0|T=0])$ — селекционное смещение: разница между группами в отсутствие воздействия. Оно систематическое, а не случайное, и именно его обнуляет рандомизация.

**Постановка проблемы:** \
Влияние некоторых переменных невозможно проверить, используя A/B-тестирование. Например:

- Некоторые эксперименты неэтичны. Нельзя заставить студенток употреблять алкоголь, чтобы увидеть, как это повлияет на их успеваемость в университете.
- Проведение ряда других экспериментов запрещено юридическими ограничениями. Например, вы не можете устанавливать разные цены на один и тот же продукт.
- Также существуют ситуации, когда провести эксперимент невозможно. Если происходит ребрендинг, запускать товар на две достаточно большие непересекающиеся группы может быть дорого.


В этом случае исследовательнице приходится работать с тем, что есть.

Что может помочь оценить влияние признака? **Линейная регрессия!**

**Используемый датасет:** [UCI Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance) — анкеты и оценки 395 учащихся двух португальских школ; итоговая оценка по математике `G3` стоит по шкале от 0 до 20.


Выясним, как готовность получить высшее образование влияет на итоговую оценку по математике. А именно попробуем построить модель:

$$\text{math\_grade} = \beta_0 + \beta_1 \cdot \text{higher} + \varepsilon,$$
где:
- $\beta_0$ — свободный член
- $\beta_1$ — коэффициент при признаке: величина, на которую меняется прогноз оценки, когда готовность получить высшее образование переходит с 0 на 1
- $\text{higher}$ — готовность получить высшее образование
- $\varepsilon$ — случайный шум

In [ ]:
data = pd.read_csv('https://github.com/aiedu-courses/all_datasets/raw/main/student-mat.csv',
                   sep=';')
data.head()

In [ ]:
model = smf.ols('G3 ~ higher', data=data).fit()
model.summary().tables[1]

**Что значит каждый коэффициент?**
- coef — при Intercept значение целевой переменной, когда все признаки равны 0; при признаке — изменение прогноза, когда признак переходит с 0 на 1;
- std err — стандартная ошибка оценки коэффициента;
- P>|t| — p-value: так как здесь оно меньше 0.05, можно сказать, что готовность к высшему образованию значимо связана с прогнозом.

Но на этом информация из регрессии не заканчивается!

In [ ]:
model.summary()

Здесь:
- Когда была обучена модель;
- Количество наблюдений;
- Метод обучения OLS;
- Качество регрессии: $R^2, R^2_{adj}$;
- F-statistic: значение статистики теста Фишера с нулевой гипотезой «регрессия не значима», то есть совсем не информативна;
- Prob (F-statistic): p-value этого теста — вероятность получить такое же или большее значение F-статистики, если регрессия неинформативна;
- Log-Likelihood: логарифм правдоподобия по выборке;
- AIC, BIC: информационные критерии для сравнения регрессий.

## Контроль по наблюдаемым переменным

Коэффициент $\hat\beta_1$ модели с одним признаком — это разность средних оценок двух групп, а в ней, как мы видели выше, смешаны эффект и селекционное смещение. Если учащиеся, которые собираются получать высшее образование, отличаются от остальных чем-то еще — временем на учебу, числом прошлых неудач, образованием родителей, — эта разница попадет в $\hat\beta_1$.

Первое средство — добавить такие переменные в регрессию как контроли. Возьмем все анкетные признаки, кроме промежуточных оценок `G1` и `G2`: их ставят в том же учебном году, они сами зависят от готовности учиться дальше, и контроль по ним отрезал бы часть эффекта, который мы хотим измерить (это медиаторы).

In [ ]:
controls = ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob',
            'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid',
            'activities', 'nursery', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
            'Walc', 'health', 'absences']
print(len(controls), 'контролей')
print(data['higher'].value_counts())

**Задание 1. Постройте OLS-модель `G3 ~ higher + <все контроли из списка controls>` и выведите коэффициент при `higher[T.yes]`, его стандартную ошибку и p-value. Как изменился коэффициент по сравнению с моделью с одним признаком?**

In [ ]:
# Ваш код здесь

## DML: остатки вместо формулы

В линейной регрессии с контролями мы сами решаем, в каком виде контроли входят в модель: линейно, без взаимодействий. DML (double/debiased machine learning, [Chernozhukov et al., 2018](https://arxiv.org/abs/1608.00060)) снимает это ограничение. Базовый сценарий — partial linear regression:

$$Y = \theta\,T + g(X) + \varepsilon,\qquad T = m(X) + \nu.$$

1. Любой ML-моделью предсказываем $E[Y\mid X]$ и $E[T\mid X]$.
2. Считаем остатки $\tilde Y = Y - \hat E[Y\mid X]$ и $\tilde T = T - \hat m(X)$.
3. Оценка эффекта — коэффициент регрессии остатков: $\hat\theta = \sum_i \tilde T_i \tilde Y_i \big/ \sum_i \tilde T_i^2$.

Чтобы переобучение вспомогательных моделей не попало в оценку, прогнозы для каждого объекта делаются моделями, которые его не видели: это cross-fitting. Ниже матрица признаков `X` собрана из тех же контролей.

In [ ]:
X = pd.get_dummies(data[controls], drop_first=True).astype(float).values
y = data['G3'].values.astype(float)
t = (data['higher'] == 'yes').astype(float).values
print(X.shape)

**Задание 2. Реализуйте DML с cross-fitting на 5 фолдах (`KFold(5, shuffle=True, random_state=0)`): $E[Y\mid X]$ — `RandomForestRegressor(n_estimators=500, min_samples_leaf=5, random_state=0)`, $E[T\mid X]$ — `RandomForestClassifier` с теми же настройками и `predict_proba`. Выведите $\hat\theta$, его стандартную ошибку и долю объектов, у которых $\hat m(X) > 0{,}95$.**

Стандартная ошибка DML: $\widehat{se} = \sqrt{\tfrac1n\sum_i \tilde T_i^2\,(\tilde Y_i-\hat\theta\tilde T_i)^2\big/n}\;\Big/\;\tfrac1n\sum_i \tilde T_i^2$.

Оформите решение функцией `dml(y, t, X)`, она понадобится еще раз.

In [ ]:
# Ваш код здесь

Посмотрите на долю объектов с $\hat m(X)$ около единицы. Не собираются получать высшее образование только 20 учащихся из 395, и для большинства остальных модель склонности уверена, что они в группе $T=1$. У таких объектов остаток $\tilde T$ почти нулевой, и оценка $\hat\theta$ держится на немногих объектах. Это нарушение позитивности, о котором говорит урок «IV и DML»: DML аккуратно считает эффект при условии, что конфаундеры измерены все и что в каждой области признаков есть объекты обеих групп.

## Инструментальная переменная

DML и регрессия с контролями убирают смещение только от тех конфаундеров, которые измерены. Если есть скрытый конфаундер — например, мотивация, которой в анкете нет, — нужен другой источник вариации воздействия: инструментальная переменная $Z$. Она должна влиять на воздействие (relevance), не влиять на исход иначе как через воздействие (exclusion) и не зависеть от скрытых конфаундеров (independence).

В этом датасете такого инструмента нет. Признаки, которые связаны с `higher`, — образование родителей, платные занятия, прошлые неудачи — влияют на оценку и напрямую, то есть нарушают exclusion, а проверить exclusion по данным нельзя.

Поэтому для IV возьмем полусинтетические данные: признаки учащихся настоящие, а воздействие, инструмент и исход сгенерированы так, что истинный эффект известен. Сюжет — как в уроке «IV и DML»: школа разыгрывает бесплатные места на подготовительный курс по лотерее ($Z$), курс ($T$) проходят чаще выигравшие и более мотивированные, мотивация ($U$) в данных не записана и сама поднимает оценку. Истинный эффект курса $\theta = 2$ балла.

In [ ]:
rng = np.random.default_rng(42)
n = len(data)
U = rng.normal(size=n)                                   # мотивация: скрытый конфаундер
Z = rng.binomial(1, 0.5, size=n)                         # лотерея: инструмент
g_x = 0.3 * (data['studytime'].values - 2) - 0.5 * data['failures'].values
T_syn = (-0.3 + 1.5 * Z + 1.0 * U + g_x + rng.normal(size=n) > 0).astype(float)
theta_true = 2.0
f_x = 0.8 * data['studytime'].values - 1.5 * data['failures'].values + 0.3 * data['Medu'].values
Y_syn = 8 + theta_true * T_syn + f_x + 2.0 * U + rng.normal(size=n)

syn = pd.DataFrame(X, columns=[f'x{j}' for j in range(X.shape[1])])
syn['Y'], syn['T'], syn['Z'] = Y_syn, T_syn, Z
x_terms = ' + '.join(f'x{j}' for j in range(X.shape[1]))
print('доля прошедших курс:', round(T_syn.mean(), 3))
print('OLS без контролей:', round(smf.ols('Y ~ T', data=syn).fit().params['T'], 3))
print('OLS с контролями: ', round(smf.ols('Y ~ T + ' + x_terms, data=syn).fit().params['T'], 3))

**Задание 3. Посчитайте IV-оценку эффекта курса двумя способами.**

1. **2SLS по шагам:** первый шаг — OLS `T ~ Z + <контроли x_terms>`; выведите F-статистику инструмента (квадрат t-статистики при `Z`). Второй шаг — OLS `Y ~ T_hat + <контроли>`, где `T_hat` — прогноз первого шага. Коэффициент при `T_hat` — оценка 2SLS.
2. **Оценка Вальда:** разность средних `Y` между выигравшими и проигравшими лотерею, деленная на разность долей прошедших курс.

Сравните обе оценки с OLS выше и с истинным $\theta = 2$.

*Стандартная ошибка из второго шага, сделанного вручную, неверна: она не учитывает, что `T_hat` сам оценен. Для интервала используйте готовую реализацию 2SLS, например `IV2SLS` из пакета `linearmodels`.*

In [ ]:
# Ваш код здесь

**Задание 4. Примените функцию `dml` из задания 2 к полусинтетическим данным дважды: с признаками `X` и с признаками `X` плюс столбец мотивации `U` (как если бы ее измерили). Сравните обе оценки с истинным $\theta = 2$ и с IV.**

In [ ]:
# Ваш код здесь

Сравните четыре оценки на полусинтетических данных. Регрессия с контролями и DML по $X$ видят одни и те же переменные и потому одинаково не видят мотивацию: от того, что вспомогательные модели стали гибкими, скрытый конфаундер не становится измеренным. IV использует только ту часть вариации курса, которую создала лотерея, а она от мотивации не зависит. DML возвращается к истинному эффекту, когда мотивация попадает в признаки.

**Выводы. Подчеркнем, что:**

- Моделирование связей «причина — следствие» ограничено наличием скрытых взаимосвязей в данных. Поэтому к любым причинно-следственным выводам, основанным на наблюдениях, следует относиться с долей скептицизма. Мы не можем быть уверены, что полученная нами оценка полностью объективна.

- Контроль по наблюдаемым переменным и DML убирают смещение только от измеренных конфаундеров; DML вдобавок требует, чтобы в каждой области признаков были объекты обеих групп.

- Инструментальная переменная справляется и со скрытым конфаундером, но держится на допущении exclusion, которое по данным не проверить.

- Имея данные наблюдений, мы можем только выдвигать гипотезы и проверять их на основе моделей.

- Регрессия из statsmodels сохраняет информацию, позволяющую анализировать построенную модель сразу после построения, без дополнительных тестов.

Спасибо за ваше время на материал!

Источники:
1. [Causal Inference for the Brave and True](https://matheusfacure.github.io/python-causality-handbook/01-Introduction-To-Causality.html)
2. [Linear regressions for causal conclusions](https://towardsdatascience.com/linear-regressions-for-causal-conclusions-34c6317c5a11)
3. [Chernozhukov et al., 2018. Double/debiased machine learning for treatment and structural parameters](https://arxiv.org/abs/1608.00060)